In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv(
    r"E:\eeg-analysis\eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv"
)

y = df["Target_Label"]

X = df.drop(columns=[
    "Participant_ID",
    "Time_Stamp",
    "Target_Label"
])


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# Check your data types
print("Data types in X:")
print(X.dtypes)
print("\nSample of X:")
print(X.head())

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns
print(f"\nCategorical columns: {list(categorical_cols)}")

# Option 1: Label Encoding for categorical variables
X_encoded = X.copy()
label_encoders = {}

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    label_encoders[col] = le

# Option 2: One-Hot Encoding (alternative approach)
# X_encoded = pd.get_dummies(X, drop_first=True)

# Now run cross-validation
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv_results = cross_validate(
    rf,
    X_encoded,  # Use encoded data
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro"
    },
    return_train_score=True
)

print("Mean CV Accuracy:", cv_results["test_accuracy"].mean())
print("Train-Test Gap:",
      cv_results["train_accuracy"].mean() -
      cv_results["test_accuracy"].mean())

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv(
    r"E:\eeg-analysis\eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv"
)

print(df.columns)

print(df.select_dtypes(include=["object"]).columns)
y = df["Target_Label"]

print(y)

X = df.drop(columns=[
    "Participant_ID",
    "Time_Stamp",
    "Target_Label",
    "Task"   # ← WAJIB
])
print(X.select_dtypes(include=["object"]).columns)
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

cv_results = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro"
    },
    return_train_score=True
)

print("Mean CV Accuracy:", cv_results["test_accuracy"].mean())
print(
    "Train-Test Gap:",
    cv_results["train_accuracy"].mean() -
    cv_results["test_accuracy"].mean()
)


In [ ]:
print(y)

----------------------------------

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ============================================================
# 1. LOAD DATA
# ============================================================
df = pd.read_csv(
    r"E:\eeg-analysis\eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv"
)

print("="*60)
print("DATA INFO")
print("="*60)
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData types:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")

# ============================================================
# 2. IDENTIFIKASI KOLOM 64 CHANNEL
# ============================================================
# Cari semua kolom yang merupakan channel EEG
# Biasanya bernama seperti: Fp1, Fp2, F3, F4, C3, C4, dll.
# Atau bisa juga Ch1, Ch2, ..., Ch64

# Exclude kolom non-EEG
non_eeg_cols = [
    "Participant_ID",
    "Time_Stamp", 
    "Target_Label",
    "Task"
]

# Pilih semua kolom numerik yang bukan kolom metadata
eeg_channels = [col for col in df.columns if col not in non_eeg_cols]

print("\n" + "="*60)
print("EEG CHANNELS DETECTED")
print("="*60)
print(f"Total channels: {len(eeg_channels)}")
print(f"Channel names: {eeg_channels}")

# ============================================================
# 3. PREPARE FEATURES & TARGET
# ============================================================
y = df["Target_Label"]
X = df[eeg_channels]

# Pastikan semua data numerik
print("\n" + "="*60)
print("FEATURE MATRIX (X)")
print("="*60)
print(f"Shape: {X.shape}")
print(f"\nData types:\n{X.dtypes.value_counts()}")
print(f"\nSample data:\n{X.head()}")

# Cek dan handle kolom kategorikal jika ada
categorical_cols = X.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print(f"\nWarning: Found categorical columns: {list(categorical_cols)}")
    print("Applying Label Encoding...")
    
    X_encoded = X.copy()
    label_encoders = {}
    for col in categorical_cols:
        le = LabelEncoder()
        X_encoded[col] = le.fit_transform(X[col])
        label_encoders[col] = le
    X = X_encoded

# ============================================================
# 4. CHECK TARGET DISTRIBUTION
# ============================================================
print("\n" + "="*60)
print("TARGET DISTRIBUTION")
print("="*60)
print(y.value_counts())
print(f"\nClass balance:\n{y.value_counts(normalize=True)}")

# ============================================================
# 5. CROSS-VALIDATION DENGAN RANDOM FOREST
# ============================================================
print("\n" + "="*60)
print("RUNNING 5-FOLD CROSS-VALIDATION")
print("="*60)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=50,
    max_depth=3,              # Kurangi kompleksitas
    min_samples_split=5,
    min_samples_leaf=3,
    max_features='sqrt',
    random_state=42
)

cv_results = cross_validate(
    rf,
    X,
    y,
    cv=cv,
    scoring={
        "accuracy": "accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "f1_macro": "f1_macro"
    },
    return_train_score=True,
    verbose=1
)

# ============================================================
# 6. DISPLAY RESULTS
# ============================================================
print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)

metrics = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

for metric in metrics:
    train_scores = cv_results[f"train_{metric}"]
    test_scores = cv_results[f"test_{metric}"]
    
    print(f"\n{metric.upper()}:")
    print(f"  Train: {train_scores.mean():.4f} (±{train_scores.std():.4f})")
    print(f"  Test:  {test_scores.mean():.4f} (±{test_scores.std():.4f})")
    print(f"  Gap:   {train_scores.mean() - test_scores.mean():.4f}")

print("\n" + "="*60)
print("DETAILED FOLD SCORES")
print("="*60)
results_df = pd.DataFrame({
    'Fold': range(1, 6),
    'Train_Acc': cv_results['train_accuracy'],
    'Test_Acc': cv_results['test_accuracy'],
    'Train_F1': cv_results['train_f1_macro'],
    'Test_F1': cv_results['test_f1_macro']
})
print(results_df)

# ============================================================
# 7. TRAIN FINAL MODEL & FEATURE IMPORTANCE
# ============================================================
print("\n" + "="*60)
print("TRAINING FINAL MODEL FOR FEATURE IMPORTANCE")
print("="*60)

rf_final = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
rf_final.fit(X, y)

# Get feature importance
feature_importance = pd.DataFrame({
    'Channel': X.columns,
    'Importance': rf_final.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nTop 20 Most Important Channels:")
print(feature_importance.head(20))

# ============================================================
# 8. SAVE RESULTS (OPTIONAL)
# ============================================================
# Uncomment jika ingin menyimpan hasil
# feature_importance.to_csv('feature_importance_64ch.csv', index=False)
# results_df.to_csv('cv_results_64ch.csv', index=False)

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)

In [ ]:
print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Test Accuracy:  {cv_results['test_accuracy'].mean():.4f} (±{cv_results['test_accuracy'].std():.4f})")
print(f"Train Accuracy: {cv_results['train_accuracy'].mean():.4f}")
print(f"Overfitting Gap: {cv_results['train_accuracy'].mean() - cv_results['test_accuracy'].mean():.4f}")
print(f"\nTest F1-Score:  {cv_results['test_f1_macro'].mean():.4f} (±{cv_results['test_f1_macro'].std():.4f})")

In [ ]:
import joblib
joblib.dump(rf_final, "rf_final_model.pkl")
print("✅ rf_final_model.pkl berhasil disimpan!")

------------------

In [ ]:
import joblib
import pandas as pd

model = joblib.load("rf_final_model.pkl")
new_data = pd.read_csv(r"E:\eeg-analysis\Memory_Recall.csv")

print("Model features:", model.feature_names_in_)
print("New data columns:", new_data.columns.tolist())
print("Model feature count:", len(model.feature_names_in_))
print("New data feature count:", new_data.shape[1])
